# Day 63 - parallelism and collective communication

Splitting a model across GPUs is a choice of which collective runs, how often, and on
how many bytes. This notebook builds the collectives on a simulated ring of ranks,
counts the traffic, and then re-derives data, tensor, expert, context and pipeline
parallelism as different call sites for the same few primitives.

## 1. The collectives

`N` is the size of the tensor and `p` the number of ranks. The formulas are the ring or
tree implementations the libraries actually use.

In [ ]:
import numpy as np
from parallelism import (Wire, time_us, ring_us, tree_us, ring_all_reduce,
                         ring_reduce_scatter, ring_all_gather, tree_all_reduce,
                         all_to_all, broadcast, collective_costs, MODEL)

for r in collective_costs(p=8, n_bytes=256 * 1024 ** 2):
    print(f"{r['name']:16} {r['bytes_formula']:>14}  {r['steps_formula']:>9} steps"
          f"   {r['mb_per_rank']:7.1f} MB/rank")

broadcast                     N    log2(p) steps     256.0 MB/rank
reduce                        N    log2(p) steps     256.0 MB/rank
all-reduce         2(p-1)/p * N     2(p-1) steps     448.0 MB/rank
reduce-scatter      (p-1)/p * N        p-1 steps     224.0 MB/rank
all-gather          (p-1)/p * N        p-1 steps     224.0 MB/rank
all-to-all          (p-1)/p * N        p-1 steps     224.0 MB/rank
point-to-point                N          1 steps     256.0 MB/rank


In [ ]:
rng = np.random.default_rng(0)
bufs = [rng.standard_normal(1 << 14).astype(np.float32) for _ in range(8)]
ref = np.sum(bufs, axis=0)

wire = Wire()
out = ring_all_reduce(bufs, wire)
print('every rank agrees :', all(np.allclose(o, ref, atol=1e-4) for o in out))
print('steps             :', wire.steps, '= 2(p-1)')
print('bytes per rank    :', f'{wire.bytes_per_rank:,}',
      '= 2(p-1)/p x', f'{bufs[0].nbytes:,}')

every rank agrees : True
steps             : 14 = 2(p-1)
bytes per rank    : 114,688 = 2(p-1)/p x 65,536


## 2. Ring or tree

Small messages are latency-bound (count the hops), large ones are bandwidth-bound (count
the bytes), so the libraries switch implementation by size.

In [ ]:
for nb in (4 * 1024, 64 * 1024, 1024 * 1024, 64 * 1024 ** 2, 1024 ** 3):
    r, t = ring_us(nb), tree_us(nb)
    print(f'{nb:12,} bytes   ring {r:9.1f} us   tree {t:9.1f} us   '
          f"-> {'ring' if r < t else 'tree'}")

       4,096 bytes   ring      28.0 us   tree      12.1 us   -> tree
      65,536 bytes   ring      28.3 us   tree      13.0 us   -> tree
   1,048,576 bytes   ring      32.6 us   tree      27.7 us   -> tree
  67,108,864 bytes   ring     321.6 us   tree    1018.6 us   -> ring
1,073,741,824 bytes   ring    4725.6 us   tree   16118.1 us   -> ring


## 3. Data parallelism, and why the all-reduce is the whole thing

The average of the per-rank gradients *is* the full-batch gradient, because the loss is
a mean. Four ranks with a quarter of the batch each land on the same weights.

In [ ]:
from parallelism import train_single, train_ddp, ddp_overlap, zero_table

s_loss, s_par = train_single()
d_loss, d_par = train_ddp(p=4)
print(f'one device : loss {s_loss:.6f}')
print(f'4 ranks    : loss {d_loss:.6f}')
print('max weight difference:',
      f'{max(float(np.abs(a - b).max()) for a, b in zip(s_par, d_par)):.2e}')

one device : loss 0.014105
4 ranks    : loss 0.014105
max weight difference: 1.19e-07


In [ ]:
for r in ddp_overlap():
    print(f"{r['buckets']:3} buckets of {r['bucket_mb']:7.0f} MB   "
          f"all-reduce {r['comm_ms_each']:6.2f} ms each   "
          f"exposed at the end {r['exposed_ms']:5.2f} ms")

  1 buckets of    1024 MB   all-reduce   4.73 ms each   exposed at the end  4.73 ms
  4 buckets of     256 MB   all-reduce   1.20 ms each   exposed at the end  1.20 ms
 25 buckets of      41 MB   all-reduce   0.22 ms each   exposed at the end  0.22 ms


## 4. ZeRO and FSDP

Adam in mixed precision keeps 16 bytes per parameter - eight times the weights. Sharding
that across ranks is what makes a 70B model trainable at all.

In [ ]:
for r in zero_table(p=8):
    print(f"{r['stage']:30} {r['gb_per_rank']:7.0f} GB/rank   "
          f"{r['comm_gb']:7.1f} GB wire   {r['comm_vs_ddp']:.2f}x DDP")

DDP (no sharding)                 1043 GB/rank     228.2 GB wire   1.00x DDP
ZeRO-1 (optimizer)                 359 GB/rank     228.2 GB wire   1.00x DDP
ZeRO-2 (+ gradients)               244 GB/rank     228.2 GB wire   1.00x DDP
ZeRO-3 / FSDP (+ parameters)       130 GB/rank     342.3 GB wire   1.50x DDP


## 5. Tensor parallelism

Column-split the first matrix, row-split the second, and one all-reduce closes the
block. Drop the all-reduce and nothing crashes - the numbers are just wrong.

In [ ]:
from parallelism import mlp_block, tensor_parallel_mlp, tp_traffic, sequence_parallel_bytes

rng = np.random.default_rng(1)
x = rng.standard_normal((64, 256)).astype(np.float32)
w1 = (rng.standard_normal((256, 1024)) / 16).astype(np.float32)
w2 = (rng.standard_normal((1024, 256)) / 32).astype(np.float32)
y = mlp_block(x, w1, w2)
print('with the all-reduce   :',
      f'{np.abs(tensor_parallel_mlp(x, w1, w2, 4) - y).max():.2e}')
print('without it            :',
      f'{np.abs(tensor_parallel_mlp(x, w1, w2, 4, skip_all_reduce=True) - y).max():.2f}')
t = tp_traffic(8192, p=8)
print(f"\n8-way TP, 8,192 tokens: {t['all_reduces']} all-reduces of "
      f"{t['per_all_reduce_mb']:.0f} MB = {t['total_gb']:.0f} GB per pass")

with the all-reduce   : 1.43e-06
without it            : 2.57

8-way TP, 8,192 tokens: 160 all-reduces of 224 MB = 35 GB per pass


## 6. Expert parallelism

The weights stay, the tokens travel: one all-to-all out, one back.

In [ ]:
from parallelism import moe_reference, expert_parallel, ep_traffic

rng = np.random.default_rng(2)
xm = rng.standard_normal((64, 128)).astype(np.float32)
experts = [((rng.standard_normal((128, 256)) / 11).astype(np.float32),
            (rng.standard_normal((256, 128)) / 16).astype(np.float32)) for _ in range(8)]
route = rng.integers(0, 8, size=64)
print('matches the single-device MoE:',
      f'{np.abs(expert_parallel(xm, experts, route, 4) - moe_reference(xm, experts, route)).max():.2e}')
e = ep_traffic(8192, p=8)
print(f"8-way EP, 8,192 tokens, top-8: {e['total_gb']:.0f} GB per pass "
      f"({e['fp8_total_gb']:.0f} GB if the dispatch is fp8)")

matches the single-device MoE: 0.00e+00
8-way EP, 8,192 tokens, top-8: 140 GB per pass (70 GB if the dispatch is fp8)


## 7. Context parallelism

Each rank owns a slice of the keys and values, computes a partial attention plus its
log-sum-exp, and the ranks merge with the Day 52 identity - the same algebra as
FlashAttention, except the halves are on different GPUs.

In [ ]:
from parallelism import attention, context_parallel_attention, kv_cache_gb, kv_layouts

rng = np.random.default_rng(3)
q = rng.standard_normal((32, 128)).astype(np.float32)
k = rng.standard_normal((8192, 128)).astype(np.float32)
v = rng.standard_normal((8192, 128)).astype(np.float32)
full = attention(q, k, v)
for p in (2, 4, 8):
    print(f'{p} ranks: {np.abs(context_parallel_attention(q, k, v, p) - full).max():.2e}')

print(f'\nKV cache for 64 requests x 128K tokens: {kv_cache_gb():,.0f} GB')
for name, gb, why in kv_layouts(p=8):
    print(f'  {name:34} {gb:9,.0f} GB/rank   {why}')

2 ranks: 9.71e-17
4 ranks: 6.81e-17
8 ranks: 6.94e-17

KV cache for 64 requests x 128K tokens: 2,560 GB
  TP, MHA/GQA (KV split by head)           320 GB/rank   heads divide by p
  TP, MLA (one latent per token)         2,560 GB/rank   the latent cannot be split by head
  DP attention (requests split)            320 GB/rank   each rank owns whole requests
  context parallel (sequence split)        320 GB/rank   each rank owns a key range


## 8. Pipeline parallelism and the placement table

Pipeline moves almost nothing and idles instead. Lining the axes up by bytes is what
decides which of them is allowed to cross a network.

In [ ]:
from parallelism import pipeline_bubble, pipeline_traffic, placement_table

for r in pipeline_bubble(stages=8)[:5]:
    print(f"{r['micro']:3} micro-batches   bubble {r['bubble']:6.1%}   "
          f"utilisation {r['util']:6.1%}")
print()
for r in placement_table():
    print(f"{r['axis']:24} {r['gb']:8.2f} GB   in-node {r['in_node_ms']:7.1f} ms   "
          f"cross-node {r['cross_node_ms']:8.1f} ms")

  1 micro-batches   bubble  87.5%   utilisation  12.5%
  2 micro-batches   bubble  77.8%   utilisation  22.2%
  4 micro-batches   bubble  63.6%   utilisation  36.4%
  8 micro-batches   bubble  46.7%   utilisation  53.3%
 16 micro-batches   bubble  30.4%   utilisation  69.6%

tensor parallel (TP)        35.00 GB   in-node    98.4 ms   cross-node    774.0 ms
expert parallel (EP)       140.00 GB   in-node   378.0 ms   cross-node   3017.7 ms
context parallel (CP)       17.64 GB   in-node    48.5 ms   cross-node    384.3 ms
pipeline parallel (PP)       0.12 GB   in-node     0.3 ms   cross-node      2.7 ms
data parallel (DDP)        228.17 GB   in-node   612.5 ms   cross-node   4900.1 ms


## 9. Five ways this goes wrong without an error message

In [ ]:
from parallelism import (forgot_to_average, mismatched_order, reduction_order,
                         straggler, missing_all_reduce)

for r in forgot_to_average(steps=(1, 3, 5)):
    print(f"{r['steps']:2} steps   averaged {r['averaged']:8.4f}   summed {r['summed']:,.1f}")
m = mismatched_order()
print(f"\nmismatched order: rank 0 waits at '{m['waiting_on'][0]}', "
      f"rank 1 at '{m['waiting_on'][1]}' - forever")
ro = reduction_order()
print(f"reduction order : {ro['differing']:,} of {ro['n']:,} values differ between ring "
      f"and tree, max {ro['ring_vs_tree']:.1e}; the ring repeats exactly: {ro['ring_repeatable']}")
for r in straggler():
    print(f"{r['ranks']:5} ranks   a rank hiccups on {r['any_hiccup_pct']:5.1f}% of steps"
          f"   step is {r['loss_pct']:5.1f}% slower than the average rank")
ma = missing_all_reduce()
print(f"dropped all-reduce: shape still fine ({ma['shape_ok']}), error "
      f"{ma['rel_dropped']:.0%} of the output scale")

 1 steps   averaged   0.6129   summed 6.8
 3 steps   averaged   0.3251   summed 70,331.4
 5 steps   averaged   0.2362   summed 17,586,749,440.0

mismatched order: rank 0 waits at 'grad', rank 1 at 'loss' - forever
reduction order : 32,459 of 65,536 values differ between ring and tree, max 9.8e-04; the ring repeats exactly: True
    8 ranks   a rank hiccups on   7.7% of steps   step is   5.2% slower than the average rank
   64 ranks   a rank hiccups on  47.4% of steps   step is  23.7% slower than the average rank
  512 ranks   a rank hiccups on  99.4% of steps   step is  49.3% slower than the average rank
 4096 ranks   a rank hiccups on 100.0% of steps   step is  50.4% slower than the average rank
dropped all-reduce: shape still fine (True), error 77% of the output scale


## 10. Assertions

In [ ]:
from parallelism import check
check()
print('all assertions passed')

all assertions passed
